# **Dask on Strato — Setup Guide**

**This guide picks up where the [CLAAUDIA getting-started guide](https://hpc.aau.dk/strato/getting-started/) leaves off. By the end of that guide you have a running Ubuntu VM and can SSH into it. This guide takes you from there to a working multi-VM Dask cluster.**

****VPN required if off-campus.** Strato is only reachable from the AAU network. On campus (group room WiFi or wired) you can reach Strato directly — no VPN needed. Working from home or elsewhere: connect via the AAU VPN first.**

## **Part 1 — Configure the Head Node**

### **1.1 Open the extra ports in your security group**


**Dask needs two ports in addition to SSH:**

1. **In the Strato dashboard, go to **Network → Security Groups.****
2. **Click **Manage Rules** next to your security group.**
3. **Add two rules — click **Add Rule** for each:**

| **Direction** | **Protocol** | **Port** | **Remote**    |
| ------------- | ------------ | -------- | ------------- |
| Ingress       | TCP          | 8787     | 0.0.0.0/0     |
| Ingress       | TCP          | 8786     | 0.0.0.0/0     |

**Port 8786 is the Dask scheduler; port 8787 is the dashboard.**

****Recommended flavor:** `AAU.CPU.b.2-4` (2 vCPUs, 4 GB RAM) for both head node and worker VMs. The CLAAUDIA getting-started guide uses `AAU.CPU.a.1-4` (1 vCPU) — resize before continuing.**

### **1.2 Resize the VM to AAU.CPU.b.2-4**

**The CLAAUDIA getting-started guide launches a `AAU.CPU.a.1-4` VM (1 vCPU). Resize it before proceeding — the head node runs both the Dask scheduler and your Python script, and 1 vCPU causes contention.**

1. ****Shut off the VM first:** in the Strato dashboard go to **Compute → Instances**, click the dropdown → **Shut Off Instance.** Wait until status is Shutoff.**
2. **Click the dropdown again → **Resize Instance**.**
3. **Select `AAU.CPU.b.2-4` (2 vCPUs, 4 GB RAM) and click Resize.**
4. **Once the status changes to **Verify Resize**, click the dropdown → **Confirm Resize**.**
5. **Start the VM again: dropdown → **Start Instance**. Wait until **Active**.**

### **1.3 SSH into your VM**

**Use the IP shown in **Compute → Instances** next to your VM (e.g. `10.92.y.x`):**

```sh
ssh -i ~/.ssh/my_ssh_key ubuntu@10.92.y.x
```

**All commands from here until Part 3 are run **on the VM**, not on your laptop.**

### **1.4 Install Miniconda**
```sh
wget https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh
bash Miniconda3-latest-Linux-x86_64.sh -b -p ~/miniconda3
~/miniconda3/bin/conda init bash
source ~/.bashrc
```
**Verify it worked:**
```sh
conda --version
```

### **1.5 Create the Python environment**
```sh
conda create -n nsc python=3.11 "dask[distributed]" numba numpy matplotlib -c conda-forge -y
conda activate nsc
```
**If you see a CondaToSNonInteractiveError about Terms of Service, run these two lines exactly as shown, then re-run the conda create command above:**
```sh
conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r
```
**This takes a couple of minutes. When it finishes, verify the key packages. Make sure your prompt shows `(nsc)` before running — if it still shows `(base)`, `run conda activate nsc` first:**
```sh
python -c "import dask, numba, numpy; print(dask.__version__, numba.__version__, numpy.__version__)"
```

### **1.6 Adapt your L06 code for the cluster**

**Three small changes are needed before your code will work on the cluster:**

#### 1. ****Replace `LocalCluster` + `Client`** with a single `Client` line pointing at the scheduler:**
```python
# Remove these two lines:
# cluster = LocalCluster(n_workers=8, threads_per_worker=1)
# client = Client(cluster)

# Replace with:
client = Client("tcp://10.92.y.x:8786")
```
**A plain `Client()` with no argument starts a new local scheduler instead of connecting to the cluster.**

#### 2. ****Remove cluster.close()** — you do not own the cluster, only the client:**
```python
client.close()   # keep this
# cluster.close()  # remove this
```
    
### 3. ****Warm up workers** before timing. If your L06 code only warmed up locally, add a `client.run` call:**
```python
client.run(lambda: your_numba_function(warmup_args))
```
**This ensures Numba is JIT-compiled on each worker before the benchmark starts.**


**Make these changes on your laptop, then copy the file to the VM.**

****Option A — git clone (recommended if your code is on GitHub/GitLab):****

Commit and push the changes, then on the VM:

```sh
git clone https://github.com/YOUR_USERNAME/YOUR_REPO.git
```

****Option B — copy from your laptop:****

```sh
scp -i ~/.ssh/my_ssh_key mandelbrot_dask.py ubuntu@10.92.y.x:~/
```

### **1.7 Test locally**

**Run a quick sanity check on the VM to confirm Numba and Dask work before snapshotting:**
```sh
conda activate nsc
python -c "
from dask.distributed import LocalCluster, Client
import dask
cluster = LocalCluster(n_workers=2, threads_per_worker=1)
client = Client(cluster)
result = dask.delayed(sum)(range(1000))
print(dask.compute(result))
client.close(); cluster.close()
"
```
**Expected output (some startup log lines may appear before it):**
```sh
(499500,)
```
**If you see this, Dask is working correctly. Ignore any warnings about dashboard not being available or deprecated APIs.**

## **Part 2 — Snapshot and Launch Worker VMs**

### **2.1 Create a snapshot**
**Shut down the VM before snapshotting to ensure a consistent disk image. From the terminal on the VM:**

```sh
sudo shutdown -h now
```

**Your SSH connection will drop — that is expected. Wait 30 seconds, then proceed in the Strato dashboard.**

****Do the snapshot from the Strato dashboard, not the terminal.****

1. **Go to **Compute → Instances**.**
2. **Confirm the VM status is **Shutoff**.**
3. **Find your VM, click the dropdown arrow on the right, choose **Create Snapshot**.**
4. **Name it something like `nsc-dask-node` and click **Create Snapshot**.**

**Wait for the snapshot to finish (status changes to **Active** under **Compute → Images**). This takes 1–3 minutes.**

**The snapshot captures the entire disk — Miniconda, your conda environment, and your code — so all worker VMs will be byte-for-byte identical to the head node. This prevents version mismatch errors.**


### **2.2 Launch worker VMs from the snapshot**

1. **Go to **Compute → Images**, find your `nsc-dask-node` snapshot.**
2. **Click **Launch** (not the image in the public list — your own snapshot under **Project**).**
3. **Configure each worker VM:**
    - ****Instance Name:** `nsc-worker-1` (repeat for `nsc-worker-2`, etc.)**
    - ****Source:** your snapshot (already selected)**
        - ****Flavor:** `AAU.CPU.b.2-4` (2 vCPUs, 4 GB RAM) — same as head node; launch 2–3 worker VMs for 4–6 total workers. Use `AAU.CPU.d.4-12` if you want 4 workers per VM (2 VMs = 8 workers total)**
    - ****Network:** same as head node**
    - ****Security Group:** same group (ports 22, 8786, 8787 already open)**
    - ****Key Pair:** same key**
    - ****No floating IP needed** on worker VMs**
4. **Launch. Repeat to create 2–3 worker VMs.**

**Note the **internal IP** of each worker from the Instances list — you will need them shortly.**

**Before continuing, **start the head node again**: go to **Compute → Instances**, find the head node (status: Shutoff), click the dropdown → **Start Instance**. Wait until status is **Active**, then SSH back in.**

## **Part 3 — Start the Dask Cluster**

### **3.1 Note the head node's IP**
**In the Strato dashboard under **Compute → Instances**, note the IP of your head node (e.g. `10.92.y.x`). This is the **scheduler address** that workers will connect to. Use this same IP in all subsequent steps.**

### **3.2 Start the scheduler on the head node**

**SSH into the head node. Use tmux so the scheduler keeps running even if your SSH connection drops:**
```sh
tmux new -s dask
conda activate nsc
dask scheduler
```
The scheduler prints several log lines; look for:
```sh
Scheduler at:  tcp://10.92.y.x:8786
dashboard at:  http://10.92.y.x:8787/status
```

****Expected warnings — ignore these:** - `FutureWarning: dask-scheduler is deprecated` — harmless, the command still works. - `To route to workers diagnostics ... install jupyter-server-proxy` — not needed for our use.**

**To detach from tmux without stopping the scheduler: press `Ctrl-B`, then `D`. To reattach later: `tmux attach -t dask`.**

### **3.3 Start a worker on a VM**

**These steps are the same for every worker VM.**

**Open a terminal on your laptop and SSH into the worker VM (e.g. `10.92.y.x`):**
```sh
ssh -i ~/.ssh/my_ssh_key ubuntu@10.92.y.x
```
**Then on the VM:**
```sh
conda activate nsc
dask worker 10.92.y.x:8786 --nworkers -1 --nthreads 1
```
**Replace `10.92.y.x` with your **head node's IP** (from step 3.1).**

**`--nworkers -1` starts one worker process per available vCPU — so a `AAU.CPU.b.2-4` VM contributes 2 worker processes. `--nthreads 1` gives each process a single thread, which is correct for Numba.**

**Within a few seconds the scheduler window will print:**
```sh
Register worker <Worker ...>
```
**confirming the worker has connected. The dashboard Workers tab also updates immediately.**

****Experiment 2 (worker scaling):** start with one worker VM connected and run the benchmark, then SSH into a second VM, start its worker, and re-run. Each time you add a VM you increase the worker count by 2 (for the `AAU.CPU.b.2-4` flavor). Record wall time after each addition.**


## **Part 4 — Adapt Your Code and Run on the Head Node**

### **4.1 SSH into the head node**
```sh
ssh -i ~/.ssh/my_ssh_key ubuntu@10.92.y.x
conda activate nsc
```
**If you prefer to work in a Jupyter notebook instead of running a script directly, see the Appendix: [Jupyter Notebook at the end of this guide](https://www.moodle.aau.dk/mod/page/view.php?id=2076079#appendix-jupyter-notebook).**


### **4.2 Run your script**
```sh
python mandelbrot_dask.py
```
**The output should include a line like:**
```sh
<Client: 'tcp://10.92.y.x:8786' processes=6 threads=6, memory=11.46 GiB>
```

### **4.3 Check the dashboard**

**From your laptop browser (VPN required if off-campus):**

http://10.92.y.x:8787

**You should see all workers listed under the **Workers** tab and tasks executing in real time.**

### **4.4 Smoke test**

**Before benchmarking, verify all workers have the same package versions:**
```python
versions = client.run(lambda: __import__('dask').__version__)
print(versions)   # all values must be identical
```
**If any worker shows a different version, delete the mismatched VM, re-launch from the snapshot, and reconnect.**

## **Part 5 — Benchmark**

**Use **N=4096** or **N=8192** for meaningful speedup — at 1024×1024 scheduling overhead dominates and speedup will be low.**

****Head node RAM:** `dask.compute()` assembles the full result on the head node. A 4096×4096 float64 grid is ~128 MB — fine on a small VM. At 8192×8192 it is ~512 MB, and at 16384×16384 it is ~2 GB. If you want to go beyond 8192, resize the head node to a larger flavor (e.g. `AAU.CPU.b.4-8`) before running, otherwise the process may be killed by the OS. Workers do not need resizing — only the head node collects the result.**

**Record your results in the performance tracker — include a Numba single-core run at the same resolution so speedup can be computed.**

## **Part 6 — Clean Up When Done**

**Running VMs consume your quota even when idle.**

1. ****Worker VMs:** Delete them — you can always re-launch from the snapshot in minutes. In **Compute → Instances**, select each worker, click **Delete Instances**.**
2. ****Head node:** You have two options:**
    - ****Shelve (recommended):** frees the vCPU/RAM quota but keeps the disk. Resume later with Unshelve — faster than re-launching. Click the dropdown → **Shelve Instance**.**
    - ****Delete:** removes everything. Only do this if you are certain you no longer need it. Your snapshot still exists and you can re-launch from it.**
3. ****Snapshot:** Keep it. Snapshots use storage quota (not compute quota) and let you rebuild the cluster in minutes next time.**


## **Troubleshooting**

****Worker won't connect to scheduler****
- **Check that port 8786 is open in the security group (Part 1.1).**
- **Confirm you are using the head node's IP from the Instances list (e.g. `10.92.y.x`).**
- **Check the scheduler is still running: `tmux attach -t dask` on the head node.**

****Dashboard not loading in browser****
- **Confirm port 8787 is open in the security group.**
- **Confirm VPN is connected.**
- **Try http://10.92.y.x:8787 (not https).**

****SSH connection drops / scheduler stops****
- **Always use `tmux` for the scheduler (Part 3.2). If you forgot and it stopped, just re-run `dask scheduler` and reconnect the workers.**

****VPN not working at AAU****
- **Use your laptop as a mobile hotspot (share from your phone).**
- **Connect both your laptop and any physical co-worker laptops to that hotspot.**
- **Use internal IPs as normal — devices on the same hotspot subnet can reach each other directly.**
- **This also works if you want to form a temporary cluster using classmates' laptops as additional workers: each person runs `dask worker YOUR_IP:8786` on their own machine.**

****Version mismatch between workers****
- **Always launch workers from the snapshot, not from the base Ubuntu image.**
- **Run the smoke test (Part 4.5) to catch mismatches before benchmarking.**

## **Appendix: Jupyter Notebook**

**If you prefer working in a notebook rather than running a script, you can run Jupyter on the head node and tunnel it to your laptop browser.**

****On the head node:****
```sh
conda activate nsc
jupyter notebook --no-browser --port=8888
```
****On your laptop (in a separate terminal):****
```sh
ssh -i ~/.ssh/my_ssh_key -L 8888:localhost:8888 ubuntu@10.92.y.x -N
```
**Then open http://localhost:8888 in your browser.**

**The three code changes in Part 4.2 apply equally to a notebook — replace the `LocalCluster` cell with `client = Client("tcp://10.92.y.x:8786")` and remove any `cluster.close()` call.**